In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00


In [ ]:
#koneksi dgn drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cat /content/drive/MyDrive/enrichment/val/labels/-mn_jpeg.rf.d9bd3e26fcf3257c4bc8975361f5dc6d.txt #pengecekan label

1 0.46328125 0.3890625 0.9234375 0.778125

In [ ]:
!unzip "/content/ENRICMENT.v1i.yolov8.zip" -d "/content/drive/MyDrive/enrichment/"

In [ ]:
import os
import random
import shutil
from glob import glob

# ====== PATH UTAMA ======
base_path = "/content/drive/MyDrive/enrichment"
train_images_path = os.path.join(base_path, "train/images")
train_labels_path = os.path.join(base_path, "train/labels")

val_images_path = os.path.join(base_path, "val/images")
val_labels_path = os.path.join(base_path, "val/labels")

# ====== BUAT FOLDER VAL ======
os.makedirs(val_images_path, exist_ok=True)
os.makedirs(val_labels_path, exist_ok=True)

# ====== AMBIL SEMUA FILE ======
image_files = glob(train_images_path + "/*.jpg")

# Pisahkan berdasarkan kelas dari file label
safetyhelmet_files = []
nohelmet_files = []

for img_path in image_files:
    filename = os.path.basename(img_path)
    label_path = os.path.join(train_labels_path, filename.replace(".jpg", ".txt"))

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            content = f.read()
            if content.startswith("0"):  
                safetyhelmet_files.append(filename)
            elif content.startswith("1"):  
                nohelmet_files.append(filename)

# ====== HITUNG 20% ======
val_safety = int(len(safetyhelmet_files) * 0.2)
val_nohelmet = int(len(nohelmet_files) * 0.2)

print("Safetyhelmet val:", val_safety)
print("Nohelmet val:", val_nohelmet)

# ====== RANDOM PILIH ======
random.shuffle(safetyhelmet_files)
random.shuffle(nohelmet_files)

val_safety_files = safetyhelmet_files[:val_safety]
val_nohelmet_files = nohelmet_files[:val_nohelmet]

# ====== PINDAHKAN FILE ======
def move_files(file_list):
    for file in file_list:
        shutil.move(
            os.path.join(train_images_path, file),
            os.path.join(val_images_path, file)
        )
        shutil.move(
            os.path.join(train_labels_path, file.replace(".jpg", ".txt")),
            os.path.join(val_labels_path, file.replace(".jpg", ".txt"))
        )

move_files(val_safety_files)
move_files(val_nohelmet_files)

print("Split selesai!")

# ====== BUAT DATA.YAML ======
yaml_content = f"""
train: {base_path}/train/images
val: {base_path}/val/images

nc: 2
names: ['nohelmet', 'safetyhelmet']
"""

with open(os.path.join(base_path, "data.yaml"), "w") as f:
    f.write(yaml_content)

print("data.yaml berhasil dibuat!")

Safetyhelmet val: 28
Nohelmet val: 229
Split selesai!
data.yaml berhasil dibuat!


In [ ]:
model.predict(
    source="/content/drive/MyDrive/enrichment/val/images/WhatsApp-Image-2026-01-30-at-19-20-09-2-_jpeg.rf.5453f7aae10fb90108a5404d0ce3719d.jpg",
    conf=0.5,
    save=True,
    verbose=False
) #uji coba prediksi pada satu gambar manual

Results saved to /content/runs/detect/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'safetyhelmet', 1: 'nohelmet'}
 obb: None
 orig_img: array([[[221, 147,  89],
         [221, 147,  89],
         [221, 147,  89],
         ...,
         [222, 149,  91],
         [222, 149,  91],
         [222, 149,  91]],
 
        [[221, 147,  89],
         [221, 147,  89],
         [221, 147,  89],
         ...,
         [222, 149,  91],
         [222, 149,  91],
         [222, 149,  91]],
 
        [[221, 147,  89],
         [221, 147,  89],
         [221, 147,  89],
         ...,
         [222, 149,  91],
         [222, 149,  91],
         [222, 149,  91]],
 
        ...,
 
        [[ 78, 101, 123],
         [131, 154, 176],
         [141, 164, 186],
         ...,
         [ 56,  43,  41],
         [ 56,  43,  41],
         [ 55,  42,  40]],
 
        [[109, 133, 157],
         [111, 135, 159],
         [109, 133, 157],
         ..

In [ ]:
python -c "from ultralytics import YOLO; YOLO('best.pt')(source=0, conf=0.75, show=True)"
#uji coba secara real-time menggunakan webcam
#run in terminal

In [ ]:
%%writefile /content/drive/MyDrive/enrichment/data.yaml #buat file data.yaml untuk konfigurasi dataset

train: /content/drive/MyDrive/enrichment/train/images
val: /content/drive/MyDrive/enrichment/val/images

nc: 2
names: ['nohelmet', 'safetyhelmet']

Overwriting /content/drive/MyDrive/enrichment/data.yaml


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/drive/MyDrive/enrichment/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/enrichment",
    name="helmet_k3_model"
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/enrichment/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_k3_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bdc860cade0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804